# 14. Multi-Target UniProt Metadata

This notebook collects reliable biological metadata for each project target using UniProt.

Goal:
- Confirm each target has the correct protein/gene identity.
- Collect target function summaries.
- Collect standard external identifiers such as Ensembl, ChEMBL, HGNC, and PDB links.
- Save raw UniProt API responses and processed metadata tables.

This notebook is part of the data preparation stage. It helps make the final knowledge base more trustworthy.

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "uniprot"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TARGETS_FILE = PROCESSED_DIR / "multi_target_chembl_targets.csv"

print("Project root:", PROJECT_ROOT)
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

## 1. Define Target Accessions

These are canonical human UniProt accessions for the project targets.

In [ ]:
TARGET_UNIPROT_ACCESSIONS = {
    "EGFR": "P00533",
    "ERBB2": "P04626",
    "BRAF": "P15056",
    "ALK": "Q9UM73",
    "KRAS": "P01116",
    "VEGFA": "P15692",
    "MET": "P08581",
    "PIK3CA": "P42336",
}

TARGET_DISPLAY_NAMES = {
    "EGFR": "EGFR",
    "ERBB2": "HER2 / ERBB2",
    "BRAF": "BRAF",
    "ALK": "ALK",
    "KRAS": "KRAS",
    "VEGFA": "VEGFA",
    "MET": "MET",
    "PIK3CA": "PIK3CA",
}

print("Targets:", len(TARGET_UNIPROT_ACCESSIONS))
for target_symbol, accession in TARGET_UNIPROT_ACCESSIONS.items():
    print(f"{target_symbol}: {accession}")

In [ ]:
if TARGETS_FILE.exists():
    chembl_targets_df = pd.read_csv(TARGETS_FILE)
    print("Loaded ChEMBL target file:", TARGETS_FILE)
    display(chembl_targets_df)
else:
    chembl_targets_df = pd.DataFrame({"target_symbol": list(TARGET_UNIPROT_ACCESSIONS)})
    print("ChEMBL target file not found. Continuing with configured target list only.")

## 2. Helper Functions

In [ ]:
UNIPROT_API_BASE = "https://rest.uniprot.org/uniprotkb"


def get_json(url, params=None, retries=3, pause=2):
    """GET JSON with simple retries."""
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=(10, 60))
            if response.status_code == 200:
                return response.json()
            print(f"  attempt {attempt + 1}: HTTP {response.status_code}; retrying")
        except requests.exceptions.RequestException as error:
            print(f"  attempt {attempt + 1}: {type(error).__name__}; retrying")
        time.sleep(pause)
    return None


def nested_get(value, path, default=None):
    current = value
    for key in path:
        if isinstance(current, dict):
            current = current.get(key)
        elif isinstance(current, list) and isinstance(key, int) and key < len(current):
            current = current[key]
        else:
            return default
    return default if current is None else current


def get_xrefs(record, database):
    return [
        xref.get("id")
        for xref in record.get("uniProtKBCrossReferences", [])
        if xref.get("database") == database and xref.get("id")
    ]


def join_unique(values, limit=None):
    cleaned = [str(value).strip() for value in values if value is not None and str(value).strip()]
    unique_values = list(dict.fromkeys(cleaned))
    if limit is not None:
        unique_values = unique_values[:limit]
    return " | ".join(unique_values)


def extract_function_summary(record):
    function_texts = []
    for comment in record.get("comments", []) or []:
        if comment.get("commentType") == "FUNCTION":
            for text_obj in comment.get("texts", []) or []:
                text = text_obj.get("value")
                if text:
                    function_texts.append(text)
    return " ".join(function_texts)


def extract_subcellular_locations(record):
    locations = []
    for comment in record.get("comments", []) or []:
        if comment.get("commentType") == "SUBCELLULAR LOCATION":
            for item in comment.get("subcellularLocations", []) or []:
                location = nested_get(item, ["location", "value"])
                if location:
                    locations.append(location)
    return join_unique(locations)


def extract_disease_notes(record):
    notes = []
    for comment in record.get("comments", []) or []:
        if comment.get("commentType") == "DISEASE":
            disease_name = nested_get(comment, ["disease", "diseaseId"])
            disease_desc = nested_get(comment, ["disease", "description"])
            if disease_name and disease_desc:
                notes.append(f"{disease_name}: {disease_desc}")
            elif disease_name:
                notes.append(disease_name)
    return join_unique(notes, limit=5)

## 3. Download UniProt Records

Each raw record is saved so we can reproduce the processed target metadata later.

In [ ]:
uniprot_raw = {}

for target_symbol, accession in TARGET_UNIPROT_ACCESSIONS.items():
    url = f"{UNIPROT_API_BASE}/{accession}.json"
    print(f"Fetching {target_symbol}: {accession}")
    record = get_json(url)

    if record is None:
        print(f"  failed: {target_symbol}")
        uniprot_raw[target_symbol] = {"accession": accession, "record": None, "status": "failed"}
    else:
        uniprot_raw[target_symbol] = {"accession": accession, "record": record, "status": "working"}

    time.sleep(0.25)

raw_file = RAW_DIR / "multi_target_uniprot_raw.json"
with raw_file.open("w") as f:
    json.dump(uniprot_raw, f, indent=2)

print("Saved raw UniProt responses:", raw_file)

## 4. Build Processed Target Metadata

In [ ]:
metadata_rows = []

for target_symbol, payload in uniprot_raw.items():
    record = payload.get("record")
    accession = payload.get("accession")

    if record is None:
        metadata_rows.append({
            "target_symbol": target_symbol,
            "target_display_name": TARGET_DISPLAY_NAMES.get(target_symbol, target_symbol),
            "configured_uniprot_accession": accession,
            "uniprot_accession": accession,
            "source_status": "failed",
            "source": "UniProt",
        })
        continue

    protein_name = nested_get(record, ["proteinDescription", "recommendedName", "fullName", "value"])
    alternative_names = []
    for item in nested_get(record, ["proteinDescription", "alternativeNames"], []) or []:
        alt_name = nested_get(item, ["fullName", "value"])
        if alt_name:
            alternative_names.append(alt_name)

    gene_names = []
    for gene in record.get("genes", []) or []:
        primary = nested_get(gene, ["geneName", "value"])
        if primary:
            gene_names.append(primary)
        for synonym in gene.get("synonyms", []) or []:
            value = synonym.get("value")
            if value:
                gene_names.append(value)

    metadata_rows.append({
        "target_symbol": target_symbol,
        "target_display_name": TARGET_DISPLAY_NAMES.get(target_symbol, target_symbol),
        "configured_uniprot_accession": accession,
        "uniprot_accession": record.get("primaryAccession"),
        "uniprot_id": record.get("uniProtkbId"),
        "protein_name": protein_name,
        "alternative_protein_names": join_unique(alternative_names, limit=8),
        "gene_names": join_unique(gene_names, limit=12),
        "organism": nested_get(record, ["organism", "scientificName"]),
        "taxon_id": nested_get(record, ["organism", "taxonId"]),
        "function_summary": extract_function_summary(record),
        "subcellular_locations": extract_subcellular_locations(record),
        "disease_notes": extract_disease_notes(record),
        "ensembl_ids": join_unique(get_xrefs(record, "Ensembl")),
        "chembl_ids": join_unique(get_xrefs(record, "ChEMBL")),
        "hgnc_ids": join_unique(get_xrefs(record, "HGNC")),
        "pdb_ids": join_unique(get_xrefs(record, "PDB"), limit=20),
        "pdb_count": len(get_xrefs(record, "PDB")),
        "go_ids": join_unique(get_xrefs(record, "GO"), limit=20),
        "go_count": len(get_xrefs(record, "GO")),
        "source_status": payload.get("status", "working"),
        "source": "UniProt",
        "url": f"https://www.uniprot.org/uniprotkb/{record.get('primaryAccession')}/entry",
    })

uniprot_metadata_df = pd.DataFrame(metadata_rows)
print("Metadata rows:", len(uniprot_metadata_df))
display(uniprot_metadata_df)

In [ ]:
metadata_file = PROCESSED_DIR / "multi_target_uniprot_target_metadata.csv"
uniprot_metadata_df.to_csv(metadata_file, index=False)

print("Saved:", metadata_file)
print("Rows:", len(uniprot_metadata_df))

## 5. Coverage Summary

This shows whether UniProt metadata was successfully collected for each target.

In [ ]:
coverage_df = uniprot_metadata_df[[
    "target_symbol", "target_display_name", "configured_uniprot_accession", "uniprot_accession",
    "uniprot_id", "protein_name", "organism", "ensembl_ids", "chembl_ids", "hgnc_ids",
    "pdb_count", "go_count", "source_status",
]].copy()

coverage_df["has_uniprot_record"] = coverage_df["source_status"].eq("working")
coverage_df["has_function_summary"] = uniprot_metadata_df.get("function_summary", pd.Series(dtype="object")).fillna("").astype(str).str.len() > 0
coverage_df["has_external_ids"] = coverage_df[["ensembl_ids", "chembl_ids", "hgnc_ids"]].fillna("").astype(str).agg("".join, axis=1).str.len() > 0

coverage_file = PROCESSED_DIR / "multi_target_uniprot_coverage_summary.csv"
coverage_df.to_csv(coverage_file, index=False)

print("Saved:", coverage_file)
display(coverage_df)

## 6. Final Check

You should see eight rows if all project targets were processed.

In [ ]:
print("UniProt Multi-Target Metadata Complete")
print("=" * 70)
print("Targets configured:", len(TARGET_UNIPROT_ACCESSIONS))
print("Metadata rows:", len(uniprot_metadata_df))
print("Working records:", int(coverage_df["has_uniprot_record"].sum()))
print("Files created:")
print("-", raw_file)
print("-", metadata_file)
print("-", coverage_file)

display(coverage_df)